<a href="https://colab.research.google.com/github/FarazTheAnalyst/LLM-Finetuning/blob/main/LLM-Instruction-Finetuning/Instruction_finetuning_on_domain_specific_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pretrain model(LLAMA)

Non-insruction finetuning on plain text

instruction finetuning on instruction dataset

Preference alignment

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [2]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [6]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-non-instruction.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

print("Extraction completed!")

Extraction completed!


In [7]:
model_path = "/content/checkpoint-5"

In [10]:
non_instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [11]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [12]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [17]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [19]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe has significant beneficial effects on lipid profile in patients with type 2 diabetes mellitus and that the combination is more effective than Atorvastatin alone.
Ezetimibe (10mg, 20mg) was approved for use in Canada by Health Canada as a treatment for hypercholesterolemia in January 2003.
Zocor is not currently approved for sale in any other countries outside of Canada,


we have still here

Now lets start with instruction finetuning

first of all lets inspect the data

i am starting with inbuild data

In [ ]:
from datasets import load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

In [ ]:
dataset

In [ ]:
def format_row(example):
  question = example["Context"]
  answer = example["Response"]
  example["Text"] = f"[INST] {question} [/INST] {answer}"
  return example

In [ ]:
def format_row(example):
  question = example["Context"]
  answer = example["Response"]
  example["Text"] = f"[Context] {question} [/Response] {answer}"
  return example

In [ ]:
formatted_dataset = dataset.map(format_row)

In [ ]:
formatted_dataset

In [ ]:
print(formatted_dataset[0]["Text"])

In [35]:
import pandas as pd

# Convert dataset to DataFrame
df = pd.DataFrame(dataset)

In [47]:
df.to_csv("pharma_instruction_data.csv", index=False)

In [48]:
df.to_json("pharma_instruction_data.jsonl", orient="records", lines=True)

In [51]:
from datasets import load_dataset
dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [42]:
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [52]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="/content/pharma_instruction_data.jsonl", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [46]:
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

Lets Load our Data

In [54]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="pharma_instruction_data.csv", split="train")
dataset

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 51760
})

In [55]:
def format_example(example):
  prompt = f"### Instruction: \n{example["instruction"]}\n### Input:\n{example["input"]}\n### Response:\n{example["output"]}"
  return {"text": prompt}

In [56]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [57]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 51760
})

In [59]:
dataset["text"][0]

'### Instruction: \nGive three tips for staying healthy.\n### Input:\nNone\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [60]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [61]:
def tokenize_fn(example):
  tokens = tokenizer(
      example["text"],
      truncation=True,
      padding="max_length",
      max_length=512
      )
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [62]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [63]:
from peft import  LoraConfig, get_peft_model, TaskType

In [64]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [70]:
instruction_model_lora = get_peft_model(non_instruction_model, lora_config)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [71]:
# #STEP A: Load base model
# model = AutoModelForCausalLM.from_pretrained(...)
# #STEP B: Load Stage-1 LoRA
# model = PeftModel.from_pretrained(model, non_instruction_model)
# model = model.merge_and_unload()
# # Why merge?
# # → Model clean + LoRA applied inside weights
# # → Now you can attach SECOND LoRA cleanly
# #STEP C: Attach NEW LoRA for instruction
# instruction_model_lora = get_peft_model(model, lora_config)

In [72]:
args = TrainingArguments(
    output_dir = "./tinyllama-instruction",
    num_train_epochs = 3,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate = 2e-4,
    fp16=True,
    logging_steps = 20,
    save_total_limit = 1,
    report_to = "none"
)

In [73]:
trainer = Trainer(
    model=instruction_model_lora,
    args=args,
    train_dataset=tokenized,
)

In [ ]:
trainer.train()

Step,Training Loss
20,7.372809
40,0.980093
60,0.545919
80,0.501091
100,0.537249
120,0.488907
140,0.536771
160,0.513237
180,0.467918
200,0.459345


Step,Training Loss
20,7.372809
40,0.980093
60,0.545919
80,0.501091
100,0.537249
120,0.488907
140,0.536771
160,0.513237
180,0.467918
200,0.459345


In [ ]:
model_path = "/content/tinyllama-instruction/checkpoint-3"


In [ ]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:
prompt = "Explain the mechanism of action of Metformin"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model_generate(
    **inputs,
    max_new_tokens=100,
    temperature-0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import torch

In [ ]:
# load dataset
dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv", split="train")
print(dataset)

In [ ]:
# Format dataset to Alpaca-style text
def format_example(example):
  # Build unified instruction-style prompt
  prompt = f"### Instruction: \n{example["instruction"]}\n### Input:\n{example["input"]}\n### Response:\n{example["output"]}"
  return {"text": prompt}

In [ ]:
dataset = dataset.map(format_example)
print(datast[0]["text"])

In [ ]:
# Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Tokenization iwth Response Masking
def tokenize_and_mask(example):
  text = example["text"]

  # Tokenizer full text
  enc = tokenizer(text, truncation=True, padding="max_length", max_length=512)
  inputs_ids = enc["input_ids"]

  # Find where '### Response:' starts
  response_marker = "### Response:"
  response_start = text.find(response_marker)

  if response_start != -1:
    # Token index where response begins
    response_token_start = len(tokenizer(text[:response_start])["input_ids"])
  else:
      response_token_start = 0 # if marker not found

  # Clone Labels and mask out everything before "Response"
  labels = input_ids.copy()
  labels[:response_token_start] = [-100] * response_token_start

  enc[labels] = labels
  return enc

In [ ]:
# Apply tokenization
tokenized = dataset.map(tokenize_and_mask, batched=False)
print("Tokenization + masking done.")

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
# Load base model (previously non-instructional trained)
non_instructional_trained_model = AutoModelForCausalLM.from_pretrained(
    "/content/checkpoint-5",
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
model = get_peft_model(non_instructional_model, lora_config)

In [ ]:
# Training setup
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_reate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)

In [ ]:
# Train the model
trainer.train()

In [ ]:
# Save & test the model
trainer.save_model("/content/tinyllama-instruction")
tokenizer.save_pretrained("/content/tinyllama-instruction")

In [ ]:
# Test generation
trained_model = AutoModelForCausalLM.from_pretrained("/content/tinyllama-instruction", device_map="auto")

In [ ]:
prompt = "### Instruction:\nWhat is Ezetimibe?\n### Input:\n\n### Response:\n"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))